In [ ]:
import pandas as pd
import time

In [ ]:
import io
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.service_account import Credentials

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

In [ ]:
def reitbrData (reitbrfile, reitbr_ticker):
    '''
    function to read BR reit file from excel file
    args:
    reitbr_ticker - [type]: list
    reitbrfile - [type]: excel file name
    returns
    [type]: [pandas.core.frame.DataFrame]
    '''
    # check reitbrfile file name and path
    # create reitbrfile from Funds Explorer Table in https://www.fundsexplorer.com.br/ranking
    # manually select all columns in web page copy and paste all data AS VALUES ONLY in a excel file sheet

    # scopes and paths
    scope = ['https://www.googleapis.com/auth/drive']

    # credentials json path
    jasonpath = '/home/maber/keys/googlekey.json'

    # Autenticação centralizada para os dois serviços
    credentials = Credentials.from_service_account_file(jasonpath, scopes=scope)

    # Google Drive access
    drive_service = build('drive', 'v3', credentials=credentials)

    # Busca um arquivo pelo nome exato dentro do Drive
    # Busca um arquivo pelo nome exato dentro do Drive (Corrigido)
    answer_excel = drive_service.files().list(q=f"name = '{reitbrfile}'", fields="files(id, name)").execute()
    excel_id = answer_excel.get('files', [])[0]['id']
    file = answer_excel.get('files', [])
    print("File found:", file[0]['name'], "ID:", file[0]['id'])

    # Baixar arquivo do Drive e carregar no Pandas
    request = drive_service.files().get_media(fileId=excel_id)
    file_stream = io.BytesIO()
    downloader = MediaIoBaseDownload(file_stream, request)

    done = False
    while not done:
        _, done = downloader.next_chunk()

    file_stream.seek(0)
    df = pd.read_excel(file_stream)

    return df



In [ ]:
def reitbrDataFund (url):
    '''
    function to scrape reit fundamentalist data
    args:
    url - [type]: string
    returns
    [type]: [pandas.core.frame.DataFrame]
    '''
    options = webdriver.ChromeOptions()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--log-level=3')

    # ChromeDriverManager download and uses automatic comptible
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver = webdriver.Chrome(service=Service(), options=options)
    driver.set_page_load_timeout(60)
    try:
        print('Opening the ranking page...')
        driver.get(url)
        print('Page opened; waiting for the table to render...')

        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'table.default-fiis-table__container__table'))
        )
        WebDriverWait(driver, 30).until(
            lambda d: len(d.find_elements(By.CSS_SELECTOR, 'tr[scope="row"]')) >= 10
        )
        time.sleep(2)

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.select_one('table.default-fiis-table__container__table')

        if table is None:
            raise ValueError('Table not found on the page.')

        headers = [th.get_text(' ', strip=True) for th in table.select('tr th')]
        headers = [header for header in headers if header]
        if not headers:
            headers = [th.get_text(' ', strip=True) for th in table.select('thead th')]

        rows = []
        for tr in table.select('tr[scope="row"]'):
            cells = [td.get_text(' ', strip=True) for td in tr.select('td')]
            if cells:
                rows.append(cells)

        print(f'Table found with {len(rows)} data rows.')
        if not rows:
            raise ValueError('No data rows found in the table.')

        if headers and len(headers) < len(rows[0]):
            headers = headers + [''] * (len(rows[0]) - len(headers))
        elif headers and len(headers) > len(rows[0]):
            rows = [row[:len(headers)] for row in rows]

        def _parse_value(value):
            if value is None:
                return None
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                return float(value)
            if isinstance(value, str):
                text = value.strip()
                if not text:
                    return None
                lowered = text.lower()
                if lowered in {'n/a', 'na', 'nan', 'none', '-', 'indefinido', 'indefinida'}:
                    return None

                cleaned = text.replace('R$', '').replace(' ', '')
                if '%' in cleaned:
                    if any(ch.isalpha() for ch in cleaned.replace('%', '')):
                        return text
                    numeric_text = cleaned.replace('%', '')
                else:
                    numeric_text = cleaned

                if any(ch.isalpha() for ch in numeric_text):
                    return text

                if ',' in numeric_text and '.' in numeric_text:
                    numeric_text = numeric_text.replace('.', '').replace(',', '.')
                elif ',' in numeric_text:
                    numeric_text = numeric_text.replace(',', '.')
                elif '.' in numeric_text:
                    numeric_text = numeric_text.replace('.', '')

                try:
                    parsed = float(numeric_text)
                    return parsed / 100.0 if '%' in cleaned else parsed
                except ValueError:
                    return text

            return value

        df = pd.DataFrame(rows, columns=headers[:len(rows[0])] if headers else None)
        df = df.apply(lambda col: col.map(_parse_value))

        for column in df.columns:
            parsed = df[column].map(_parse_value)
            non_missing = [value for value in parsed.dropna() if value != '']
            if non_missing and all(isinstance(value, (int, float)) and not isinstance(value, bool) for value in non_missing):
                df[column] = pd.to_numeric(parsed, errors='coerce')
            else:
                df[column] = parsed.fillna('')

        return df
    except (TimeoutException, WebDriverException, ValueError) as exc:
        print(f'Error loading the ranking from Funds Explorer: {exc}')
        return pd.DataFrame()
    finally:
        driver.quit()